This is my local simple_rag file 

In [1]:
!pip install pypdf==5.6.0
!pip install PyMuPDF==1.26.1
!pip install python-dotenv==1.1.0
!pip install langchain-community==0.3.25
!pip install langchain_openai==0.3.23
!pip install rank_bm25==0.2.2
!pip install faiss-cpu==1.11.0
!pip install deepeval==3.1.0

1) pypdf - helps to read PDF files and extract text from documents/PDF
2) PyMuPDF - this is another librar, often faster for extracting text, pages, images or metadata 
3) python-dotenv - helps to load secrets from .env file
4) langchain-commuinity- community integrations for langchain: documnet loaders, vector stores, retriever
5) langchain-openai - it is a wrapper for openAI models and embeddings (to be replaced for our usecase)
6) rank_bm25 - it is a keyword search library. useful for hybrid retrieval: semantic search plus traditional keyword search
7) faiss-cpu - local vector search library, stores embeddings and finds nearest vectors quickly
8) deepeval - evaluation framework, used to judge whether RAG answers are correct,faithful and relevant

In [ ]:
# this is the reference
import os
import sys
from dotenv import load_dotenv
from google.colab import userdata



# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable (comment out if not using OpenAI)
if not userdata.get('OPENAI_API_KEY'):
    os.environ["OPENAI_API_KEY"] = input("Please enter your OpenAI API key: ")
else:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Original path append replaced for Colab compatibility

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from helper_functions import (EmbeddingProvider,
                              retrieve_context_per_question,
                              replace_t_with_space,
                              get_langchain_embedding_provider,
                              show_context)

from evaluation.evalute_rag import evaluate_rag

from langchain.vectorstores import FAISS


In comparison to the simple_rag.ipynb, we are not gonna use the OPENAI here, instead we are gonna be using local setup:::::
- only need os and sys 
- skip dotenv and google.collab
- we use sys.executable in order to make sure that the installed library and packages are installed in the same enviourment where the notebook is running even though we have mentioned the kernal to use the same as the venv - but we don't wanted to take the chance here
-- here we have downloaded sentence-transformers ollama and langchain-huggingface

In [4]:
# for local setup we neeed 
import sys 

# check if both point to the same venv
print(sys.executable)
!which pip
!pip --version

# still in the notebooks the safer option is not to use !pip  instead this ><
!{sys.executable} -m pip install -U sentence-transformers
!{sys.executable} -m pip install ollama
!{sys.executable} -m pip install langchain-huggingface

/Users/jatin.sangwan/dev/RAG_Techniques/venv/bin/python
/Users/jatin.sangwan/dev/RAG_Techniques/venv/bin/pip
pip 26.0.1 from /Users/jatin.sangwan/dev/RAG_Techniques/venv/lib/python3.9/site-packages/pip (python 3.9)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 22.0 MB/s  0:00:0022.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 16.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 MB 8.8 MB/s  0:00:08 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 12.7 MB/s  0:00:00 17.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 11.7 MB/s  0:00:00
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 16.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 11.8 MB/s  0:00:00 16.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 

In [29]:
import os
import sys
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface  import HuggingFaceEmbeddings

import ollama



- Path- helps you work with file paths cleanly and also safer file handling
- PyPDFLoader from langchain - reads the PDF and turns it into Langchain Document Objects
    Documnet usually has: page_content, metadata(info like the page number and source files)

- from langachain_text_splitters import RecursiveCharacterTextSplitter ---> this splits text into smaller chunks, why? coz we don't want to embed the whole PDF as a giant block, RAG works better when the document is broken into meaningful chunks (CHUNKING STEP)

- from langchai_community.vectorstores import FAISS ----> FAISS is a vector DB/ search index, it stores vectors and finds the most similar vectors quickly

Path
→ find the PDF file

PyPDFLoader
→ PDF to text documents

RecursiveCharacterTextSplitter
→ documents to chunks

HuggingFaceEmbeddings
→ chunks to vectors

FAISS
→ store/search vectors

ollama
→ generate final answer locally

In [ ]:
pdf_path = Path("../data/Understanding_Climate_Change.pdf")

print(pdf_path.exists())
print(pdf_path)
print(pdf_path.resolve()) # we should be using this for PROD

True
../data/Understanding_Climate_Change.pdf
/Users/jatin.sangwan/dev/RAG_Techniques/data/Understanding_Climate_Change.pdf


In [ ]:
# here we are going to load the PDF

loader = PyPDFLoader(str(pdf_path))  # here we haven't loaded a PDF yet, we have initialized a loader that 
# knows- which file to load, how to load (logic behind it), how to convert PDF pages into langchain Documents
print(loader)
documents = loader.load()  # here we loaded the pdf using the Loader
print(documents)
print(len(documents)) # here is showed 33 Document objects means langchain created one Document per page
print(documents[0].page_content[:10])
print(documents[0].metadata)
print(type(documents[0]))
print(dir(documents[0]))

# now the next step will be::: Page Documents --> smaller text chunks

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2024-07-13T20:17:34+03:00', 'author': 'Nir', 'moddate': '2024-07-13T20:17:34+03:00', 'source': '../data/Understanding_Climate_Change.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1'}, page_content='Understanding Climate Change \nChapter 1: Introduction to Climate Change \nClimate change refers to significant, long-term changes in the global climate. The term \n"global climate" encompasses the planet\'s overall weather patterns, including temperature, \nprecipitation, and wind patterns, over an extended period. Over the past century, human \nactivities, particularly the burning of fossil fuels and deforestation, have significantly \ncontributed to climate change. \nHistorical Context \nThe Earth\'s climate has changed throughout history. Over the past 650,000 years, there have \nbeen seven cycles of glacial advance and retreat, with the abrupt end of the last ice age about \n

Why not simply “load the PDF directly”?
You can, with lower-level libraries like pypdf or PyMuPDF. But then you would need to manually handle:
opening the file
looping through pages
extracting text
creating metadata
wrapping each result into a standard object
handling edge cases
LangChain’s loader gives you a standard output shape:
Document(page_content="...", metadata={...})

In [27]:
# split the loaded PDF pages into smaller chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len  # use python len() func to measure the length of text in characters
)

# print(text_splitter)
# print(type(text_splitter))
# print(dir(text_splitter))
# print(text_splitter._chunk_size)

chunks = text_splitter.split_documents(documents)
print("Number of original page documents:", len(documents))
print("Number of chunks:", len(chunks))
print("Type of chunks:", type(chunks))
print("Type of one chunk:", type(chunks[0]))

print("\nFirst chunk preview:")
print(chunks[0].page_content[:500])

print("\nFirst chunk metadata:")
print(chunks[0].metadata)


Number of original page documents: 33
Number of chunks: 97
Type of chunks: <class 'list'>
Type of one chunk: <class 'langchain_core.documents.base.Document'>

First chunk preview:
Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the planet's overall weather patterns, including temperature, 
precipitation, and wind patterns, over an extended period. Over the past century, human 
activities, particularly the burning of fossil fuels and deforestation, have significantly 
contributed to climate change. 
Historical Context 
The Earth's c

First chunk metadata:
{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2024-07-13T20:17:34+03:00', 'author': 'Nir', 'moddate': '2024-07-13T20:17:34+03:00', 'source': '../data/Understanding_Climate_Change.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1'}


Now the next step is to CREATE EMBEDDINGS AND BUILD THE FAISS VECTOR STORE  

chunks
→ HuggingFaceEmbeddings converts each chunk into a vector
→ FAISS stores those vectors
→ vector_store becomes searchable

In [31]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(chunks, embedding_model)

print(type(vector_store))
print(vector_store)

<class 'langchain_community.vectorstores.faiss.FAISS'>


In [47]:
# Now we will query the vector store and inspect retrieved chunks

query = "What is the main cause of Climate change?"

retrieved_docs = vector_store.similarity_search(query, k=2)

print("Number of retrieved docs:", len(retrieved_docs))

print(type(retrieved_docs))  # it is a list 

# retrieved docs contain the list of document objects here and this Documnet object here refers to the instance
# of the Document class


for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Retrieved chunk {i} ---")
    print("Metadata:", doc.metadata)
    print("Content preview:")
    print(doc.page_content[:700])


    


Number of retrieved docs: 2
<class 'list'>

--- Retrieved chunk 1 ---
Metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2024-07-13T20:17:34+03:00', 'author': 'Nir', 'moddate': '2024-07-13T20:17:34+03:00', 'source': '../data/Understanding_Climate_Change.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1'}
Content preview:
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burning fossil fuels for energy releases large amounts of CO2. This includes coal, oil, and 
natural gas used for electr

Now we are going to go for the Augumentation and generation work here ::: 
Notebook Python code
→ ollama Python client
→ HTTP request to localhost:11434
→ Ollama server
→ loads/runs Gemma model
→ sends response back to notebook

In [51]:
import ollama

ollama.list()

query = "What is the main cause of climate change?"

retrieved_docs = vector_store.similarity_search(query, k=2)

context = "\n\n".join(doc.page_content for doc in retrieved_docs)
# print(context)
print(type(context))

prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""

response = ollama.chat(
    model="llama3.2",
    messages=[
        {"role": "user", "content": prompt}
    ],
)

print(response)

print(response["message"]["content"])

<class 'str'>
model='llama3.2' created_at='2026-07-29T11:56:06.955322Z' done=True done_reason='stop' total_duration=535093250 load_duration=169693708 prompt_eval_count=417 prompt_eval_duration=39182000 eval_count=18 eval_duration=322569000 message=Message(role='assistant', content='The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere.', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None
The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere.


In [62]:
# debugging and getting to know what's inside

print("Response type:", type(response))
print(response)

# print("\nMessage object:")
# print(response["message"])

# print("\nFinal answer:")
# print(response["message"]["content"])

print(response.message.content)

# I can also dump it into dict

response_dict = response.model_dump()
print(response_dict)
print(response_dict["message"]["content"])

Response type: <class 'ollama._types.ChatResponse'>
model='llama3.2' created_at='2026-07-29T11:56:06.955322Z' done=True done_reason='stop' total_duration=535093250 load_duration=169693708 prompt_eval_count=417 prompt_eval_duration=39182000 eval_count=18 eval_duration=322569000 message=Message(role='assistant', content='The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere.', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None
The primary cause of recent climate change is the increase in greenhouse gases in the atmosphere.
{'model': 'llama3.2', 'created_at': '2026-07-29T11:56:06.955322Z', 'done': True, 'done_reason': 'stop', 'total_duration': 535093250, 'load_duration': 169693708, 'prompt_eval_count': 417, 'prompt_eval_duration': 39182000, 'eval_count': 18, 'eval_duration': 322569000, 'message': {'role': 'assistant', 'content': 'The primary cause of recent climate change is the increase in greenhouse gases in the atmosphe

In [ ]:
# we will wrap that in a function here

def ask_question_with_rag(question: str) -> str:
    # here we need a context what is relevant to the question so we need to retrieve
    tempContext = vector_store.similarity_search(question, k=3)
    print(len(tempContext))
    print("This is the context that we got from the vector store", "\n\n".join(doc.page_content for doc in tempContext))
    context = "\n\n".join(doc.page_content for doc in tempContext)
    # we already have the context as well as the question that user is asking 
    # need to augment it 
    prompt = f"""
        Answer the question using only the context below.
        If the answer is not in the context, say: "I don't know based on the provided context."

        Question:
        {question}

        Context:
        {context}

        Answer in one short sentence:
    """
    # Now comes the step for generation

    response = ollama.chat(
        model="gemma4:e4B",
        messages=[{
            "role": "user",
            "content": prompt
        }]
    )

    return response.message.content

# we need to repeat all the processes in the function everytime for every question
answer = ask_question_with_rag("what is the reason for climate change")
print(answer)


3
This is the context that we got from the vector store Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burning fossil fuels for energy releases large amounts of CO2. This includes coal, oil, and 
natural gas used for electricity, heating, and transportation. The industrial revolution marked 
the beginning of a significant increase in fossil fuel consumption, which continues to rise 
today. 
Coal

Most of these climate changes are attributed to very small variations in Earth's orbit that 
change the amount of solar energy our planet rec